# 02 — RQ1 Ablation: HINTED Prompt Variants on GPT-4o-mini

**Purpose.** Run the seven-variant prompt ablation for RQ1 on the 234 curated PHP samples. This notebook covers the five variants in their HINTED form (Variants A–E), where Variants C and E include explicit category language ("SQL Injection or OS Command Injection", "CWE-89 and CWE-78 patterns"). The CLEAN re-runs of Variants C and E are produced separately by `03_run_ablation_clean.ipynb`.

**Inputs.**
- `<BASE_DIR>/final_dataset/` — produced by `01_dataset_curation.ipynb`.
- `prompts/variant_A_baseline.txt`, `prompts/variant_B_persona.txt`, `prompts/variant_C_patterns_hinted.txt`, `prompts/variant_D_cot.txt`, `prompts/variant_E_full_hinted.txt` — committed to the repository.
- `prompts/system_message.txt` — committed to the repository.
- An OpenAI API key with access to `gpt-4o-mini`.

**Outputs.**
- `<BASE_DIR>/results/rq1_ablation_results_hinted.csv` — one row per sample, with one column per variant containing the model's prediction (`Vulnerable` or `Safe`).

**Cost & runtime.** 234 samples × 5 variants = 1,170 API calls on `gpt-4o-mini`. Approximate cost: USD 0.10–0.20. Approximate runtime: 30–45 minutes depending on network latency. The script supports resume-on-interrupt (re-running picks up from where it left off).

**Note on variant naming.** This repository uses `Variant_B_Persona` to align with the paper's terminology. The original experimental code used the placeholder name `Variant_B_PERFECT`; that name is replaced here for clarity.


## 1. Setup

Edit `BASE_DIR_OVERRIDE` if you want to override the auto-located workspace directory. The OpenAI API key is read from the `OPENAI_API_KEY` environment variable; if not set, you will be prompted to enter it interactively (the key is not stored anywhere).

In [ ]:
import os
import json
import time
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

# ---- USER-EDITABLE ----
BASE_DIR_OVERRIDE = None  # e.g., Path('/content/drive/MyDrive/llm-vuln-detection-ablation')
MODEL_ID          = 'gpt-4o-mini'
TEMPERATURE       = 0.1
MAX_RETRIES       = 3
RETRY_BACKOFF_SEC = 2
# -----------------------

def find_repo_root() -> Path:
    """Locate the repository root by walking upward from the current working directory."""
    sentinels = ('README.md', 'requirements.txt', '.git')
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if any((candidate / s).exists() for s in sentinels):
            return candidate
    return cwd

REPO_ROOT         = find_repo_root()
BASE_DIR          = Path(BASE_DIR_OVERRIDE).resolve() if BASE_DIR_OVERRIDE else (REPO_ROOT / 'workspace')
DATASET_DIR       = BASE_DIR / 'final_dataset'
RESULTS_DIR       = BASE_DIR / 'results'
PROMPTS_DIR       = REPO_ROOT / 'prompts'
OUTPUT_CSV        = RESULTS_DIR / 'rq1_ablation_results_hinted.csv'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Obtain OpenAI API key from environment, fall back to interactive prompt.
api_key = os.environ.get('OPENAI_API_KEY')
if not api_key:
    from getpass import getpass
    api_key = getpass('Enter your OpenAI API key (will not be stored): ')
client = OpenAI(api_key=api_key)

print(f'REPO_ROOT     : {REPO_ROOT}')
print(f'DATASET_DIR   : {DATASET_DIR}  (exists: {DATASET_DIR.exists()})')
print(f'PROMPTS_DIR   : {PROMPTS_DIR}  (exists: {PROMPTS_DIR.exists()})')
print(f'OUTPUT_CSV    : {OUTPUT_CSV}')
print(f'MODEL_ID      : {MODEL_ID}')


## 2. Load prompt variants

Read the five HINTED prompt variants and the system message from `prompts/`. The names of the variants — `Variant_A_Baseline`, `Variant_B_Persona`, `Variant_C_Patterns`, `Variant_D_CoT`, `Variant_E_Full` — are used as column headers in the output CSV.

In [ ]:
def load_prompt(filename: str) -> str:
    """Read a prompt file and strip a single trailing newline."""
    with open(PROMPTS_DIR / filename, encoding='utf-8') as f:
        return f.read().rstrip('\n')

PROMPTS = {
    'Variant_A_Baseline':  load_prompt('variant_A_baseline.txt'),
    'Variant_B_Persona':   load_prompt('variant_B_persona.txt'),
    'Variant_C_Patterns':  load_prompt('variant_C_patterns_hinted.txt'),
    'Variant_D_CoT':       load_prompt('variant_D_cot.txt'),
    'Variant_E_Full':      load_prompt('variant_E_full_hinted.txt'),
}
SYSTEM_MESSAGE = load_prompt('system_message.txt')

for name, text in PROMPTS.items():
    print(f'{name:25s} ({len(text):3d} chars): {text[:80]}{"..." if len(text) > 80 else ""}')
print()
print(f'{"SYSTEM_MESSAGE":25s} ({len(SYSTEM_MESSAGE):3d} chars): {SYSTEM_MESSAGE[:80]}...')


## 3. Run the ablation

For each of the 234 samples, query the model five times (once per variant) and record the prediction. The output CSV is written incrementally after every sample, so the run can be safely interrupted and resumed. Per-sample, per-variant API failures are retried up to `MAX_RETRIES` times; persistent failures are recorded as `Error` in the corresponding cell.

The label of each sample (Vulnerable / Safe and the true CWE) is inferred from the parent directory name under `final_dataset/` (`CWE_89_SQLi`, `CWE_78_CmdInj`, or `Safe_Code`).

In [ ]:
def get_true_label(filepath: Path) -> tuple:
    """Infer (true_label, true_cwe) from the parent directory name."""
    folder = filepath.parent.name
    if 'Safe' in folder:
        return 'Safe', None
    if '89' in folder:
        return 'Vulnerable', 'CWE-89'
    if '78' in folder:
        return 'Vulnerable', 'CWE-78'
    raise ValueError(f'Cannot infer label from folder name: {folder}')

def predict(prompt_text: str, code_content: str) -> str:
    """Send one (prompt, code) pair to the model with retry. Return 'Vulnerable' / 'Safe' / 'Error'."""
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL_ID,
                response_format={'type': 'json_object'},
                temperature=TEMPERATURE,
                messages=[
                    {'role': 'system', 'content': SYSTEM_MESSAGE},
                    {'role': 'user',   'content': f'{prompt_text}\n\nTarget Code:\n{code_content}'},
                ],
            )
            result = json.loads(response.choices[0].message.content)
            return result.get('prediction', 'Error')
        except Exception:
            if attempt + 1 < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SEC)
            else:
                return 'Error'

# Discover samples.
all_files = sorted(DATASET_DIR.rglob('*.php'))
if not all_files:
    raise RuntimeError(f'No .php files found under {DATASET_DIR}. Run 01_dataset_curation.ipynb first.')
print(f'Total samples: {len(all_files)}')

# Resume from prior run if applicable.
if OUTPUT_CSV.exists():
    results_df = pd.read_csv(OUTPUT_CSV)
    processed = set(results_df['File_Name'].tolist())
    print(f'Resuming from existing CSV: {len(processed)} samples already processed.')
else:
    columns = ['File_Name', 'True_Label', 'True_CWE'] + list(PROMPTS.keys())
    results_df = pd.DataFrame(columns=columns)
    processed = set()

to_process = [p for p in all_files if p.name not in processed]
print(f'Samples remaining to process: {len(to_process)}')

# Main loop.
for filepath in tqdm(to_process, desc=f'Ablation on {MODEL_ID}'):
    true_label, true_cwe = get_true_label(filepath)
    code_content = filepath.read_text(encoding='utf-8', errors='ignore')

    row = {'File_Name': filepath.name, 'True_Label': true_label, 'True_CWE': true_cwe}
    for variant_name, prompt_text in PROMPTS.items():
        row[variant_name] = predict(prompt_text, code_content)

    # Incremental save: append row and rewrite CSV.
    results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)
    results_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'\nAblation complete. Results written to {OUTPUT_CSV}')


## 4. Inspect results

Quick sanity checks on the output CSV: row count, label distribution, and per-variant prediction distribution. Detailed metric computation (Accuracy, F1, Recall, Specificity, Precision, McNemar tests) is deferred to `05_metrics_and_figures.ipynb`.

In [ ]:
df = pd.read_csv(OUTPUT_CSV)

print(f'Total rows           : {len(df)}')
print(f'Expected             : 234')
print()
print('True label distribution:')
print(df['True_Label'].value_counts().to_string())
print()
print('True CWE distribution:')
print(df['True_CWE'].value_counts(dropna=False).to_string())
print()
print('Per-variant prediction distribution:')
for variant in PROMPTS.keys():
    counts = df[variant].value_counts().to_dict()
    print(f'  {variant:25s} {counts}')
